In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('historycheckpoint.csv')

C:\Users\Bui Huynh Gia Huy\AppData\Local\Temp\ipykernel_29516\2135097345.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('historycheckpoint.csv')


In [24]:
game_info = pd.read_csv('gfcheckpoint.csv')

In [3]:
data.drop(columns=['game_key'], inplace=True)

In [4]:
data.tail(5)

,app_id,game_title,date,price,regular_price,sales_percentage,shop_id,shop_name,currency
17892115,576960.0,Out Of The Box,2019-06-25,10.04,14.99,33.0,61,Steam,USD
17892116,576960.0,Out Of The Box,2019-02-12,14.99,14.99,0.0,61,Steam,USD
17892117,576960.0,Out Of The Box,2019-02-04,11.24,14.99,25.0,61,Steam,USD
17892118,576960.0,Out Of The Box,2018-07-26,14.99,14.99,0.0,61,Steam,USD
17892119,576960.0,Out Of The Box,2018-07-19,11.99,14.99,20.0,61,Steam,USD


In [27]:
print("=== CLEANING STEPS ===\n")
print(f"Initial shape: {data.shape}")

# Remove missing app_id
data_clean = data.dropna(subset=['app_id']).copy()
print(f"After removing missing app_id: {data_clean.shape}")

# Convert date to datetime
data_clean['date'] = pd.to_datetime(data_clean['date'])

data_clean['app_id'] = data_clean['app_id'].astype(str).str.rstrip('0').str.rstrip('.')

# Remove exact duplicates
initial_shape = data_clean.shape[0]
data_clean = data_clean.drop_duplicates()
print(f"Exact duplicates removed: {initial_shape - data_clean.shape[0]} rows")
print(f"After removing exact duplicates: {data_clean.shape}")

# Remove records with same app_id, date, and shop_name (keep only latest)
before_dedup = data_clean.shape[0]
data_clean = data_clean.drop_duplicates(subset=['app_id', 'date', 'shop_name'], keep='last')
print(f"Same-day price changes removed: {before_dedup - data_clean.shape[0]} rows (kept latest)")
print(f"After removing same-day duplicates: {data_clean.shape}")

# Remove rows with invalid prices (negative prices)
data_clean = data_clean[data_clean['price'] >= 0]
data_clean = data_clean[data_clean['regular_price'] >= 0]
print(f"After removing invalid prices: {data_clean.shape}")

# Remove extreme outliers (prices over 10000)
data_clean = data_clean[data_clean['price'] <= 10000]
data_clean = data_clean[data_clean['regular_price'] <= 10000]
print(f"After removing price outliers: {data_clean.shape}")

# Remove rows where sales_percentage is unreasonable (< -100 or > 100)
data_clean = data_clean[(data_clean['sales_percentage'] >= -100) & (data_clean['sales_percentage'] <= 100)]
print(f"After removing invalid sales percentages: {data_clean.shape}")

print(f"\n=== FINAL RESULT ===")
print(f"Final shape: {data_clean.shape}")
print(f"Total rows removed: {initial_shape - data_clean.shape[0]}")
print(f"\nData types:\n{data_clean.dtypes}")
print(f"\nMissing values:\n{data_clean.isnull().sum()}")

=== CLEANING STEPS ===

Initial shape: (17892120, 9)
After removing missing app_id: (17732379, 9)
Exact duplicates removed: 6268211 rows
After removing exact duplicates: (11464168, 9)
Same-day price changes removed: 563974 rows (kept latest)
After removing same-day duplicates: (10900194, 9)
After removing invalid prices: (10900194, 9)
After removing price outliers: (10899724, 9)
After removing invalid sales percentages: (10899683, 9)

=== FINAL RESULT ===
Final shape: (10899683, 9)
Total rows removed: 6832696

Data types:
app_id                      object
game_title                  object
date                datetime64[ns]
price                      float64
regular_price              float64
sales_percentage           float64
shop_id                      int64
shop_name                   object
currency                    object
dtype: object

Missing values:
app_id              0
game_title          0
date                0
price               0
regular_price       0
sales_percentage

In [28]:
# Remove games released after 2023 or with fewer than 90 records in data_clean
game_info_filtered = game_info.copy()
game_info_filtered['release_date'] = pd.to_datetime(game_info_filtered['release_date'], errors='coerce')

title_counts = data_clean['game_title'].value_counts(dropna=False)

valid_titles = (
    game_info_filtered.loc[
        game_info_filtered['release_date'].isna() | (game_info_filtered['release_date'].dt.year <= 2023),
        'game_title'
    ]
    .dropna()
    .astype(str)
)

valid_titles = set(valid_titles[valid_titles.map(title_counts).fillna(0) >= 90])

removed_game_info = len(game_info) - game_info['game_title'].astype(str).isin(valid_titles).sum()
removed_data_clean = len(data_clean) - data_clean['game_title'].astype(str).isin(valid_titles).sum()

game_info = game_info[game_info['game_title'].astype(str).isin(valid_titles)].copy()
data_clean = data_clean[data_clean['game_title'].astype(str).isin(valid_titles)].copy()

print(f"Removed from game_info: {removed_game_info}")
print(f"Removed from data_clean: {removed_data_clean}")
print(f"Remaining game_info rows: {len(game_info)}")
print(f"Remaining data_clean rows: {len(data_clean)}")

Removed from game_info: 0
Removed from data_clean: 1990171
Remaining game_info rows: 25656
Remaining data_clean rows: 8909512


In [29]:
print("=== CLEANED DATA VERIFICATION ===\n")
print(f"Sample of cleaned data:")
print(data_clean.head(10))
print(f"\nData summary:")
print(data_clean.describe())

=== CLEANED DATA VERIFICATION ===

Sample of cleaned data:
    app_id                              game_title       date  price  \
7   388390  "Glow Ball" - The billiard puzzle game 2024-04-01   2.99   
8   388390  "Glow Ball" - The billiard puzzle game 2024-03-25   0.59   
9   388390  "Glow Ball" - The billiard puzzle game 2024-01-29   2.99   
10  388390  "Glow Ball" - The billiard puzzle game 2024-01-22   0.59   
11  388390  "Glow Ball" - The billiard puzzle game 2024-01-04   2.99   
12  388390  "Glow Ball" - The billiard puzzle game 2023-12-21   0.59   
13  388390  "Glow Ball" - The billiard puzzle game 2023-11-28   2.99   
14  388390  "Glow Ball" - The billiard puzzle game 2023-11-21   0.59   
15  388390  "Glow Ball" - The billiard puzzle game 2023-10-23   2.99   
16  388390  "Glow Ball" - The billiard puzzle game 2023-10-16   0.59   

    regular_price  sales_percentage  shop_id shop_name currency  
7            2.99               0.0       61     Steam      USD  
8            2.9

In [30]:
data_clean.to_csv('historycheckpoint_cleaned.csv', index=False)